In [ ]:
from src.vanna_connector import initialize_vanna
from src.config import QDRANT_URL, QDRANT_API_KEY, OPENAI_API_KEY, OPENAI_MODEL,\
        OPENAI_API_URL, DENSE_EMBEDDING_MODEL_PATH, INTERIM_DATA_DIR, PROCESSED_DATA_DIR, DEVICE
from tqdm import tqdm
import os

/home/dima/Sber/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Initialize Vanna client

In [ ]:
# initialize configs for the database, vector store and chat model

PATH_TO_DB = os.path.join(PROCESSED_DATA_DIR, "bank_transaction_monitoring", "bank_transaction_monitoring_inline_short.sqlite.db")

# postgress example, for concrete params for different databases check VannaBase class methods connect_to_*
# postgres_config = {
#     "params": {
#         "host": "localhost",
#         "port": 5432,
#         "database": "bank_transaction_monitoring",
#         "user": "postgres",
#         "password": "postgres"
#     },
#     "type": "postgres"}


sqlite_config = {
    "params": {
        "url": str(PATH_TO_DB) # 
    },
    "type": "sqlite"}


qdrant_config = {"fastembed_model": DENSE_EMBEDDING_MODEL_PATH,
                 "url": QDRANT_URL, 
                 "api_key": QDRANT_API_KEY,
                 "device": DEVICE}

openai_config = {"api_key": OPENAI_API_KEY,
                 "model": OPENAI_MODEL,
                 "base_url": OPENAI_API_URL}

In [3]:
vanna_client = initialize_vanna(db_config=sqlite_config,
                                qdrant_config=qdrant_config,
                                openai_config=openai_config)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 5212.72it/s]
/home/dima/Sber/cursor_projects/vanna-sql/venv/lib/python3.10/site-packages/vanna/legacy/qdrant/qdrant.py:49: UserWarning: Api key is used with an insecure connection.
  self._client = QdrantClient(


# Search functionality
Allows to find top similar SQL-scripts with descriptions to the provided query

In [ ]:
query = "Сколько в среднем в месяц тратится на покупки?"
vanna_client.get_similar_question_sql(query)


# SQL generation functionality
Allows to generate SQL for the given query (uses vector store internally to retrieve relevant DDLs, scripts and documentation).

In [ ]:
query = "Сколько в среднем в месяц тратится на покупки?"
sql_script = vanna_client.generate_sql(query, allow_llm_to_see_data=True) # allow_llm_to_see_data=True - allows to make a select request to the connected database to generate SQL

print(sql_script)

vanna_client.execute_sql(sql_script) # execute generated SQL in the connected database